# Auditoría de plataformas de comercio electrónico mayorista — Posadas, Misiones
Instrumento del Anexo B de la tesis (32 indicadores, escala 0–3). Flujo: **candidatos → filtro E1–E9 → auditoría → informe HTML por dimensión**.

> Almacenamiento: solo la sesión temporal de Colab. Al final se descarga un `.zip`; si no lo descargás, se pierde.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diarce/Auditoria/blob/main/notebooks/Auditoria_Colab.ipynb)

## 1. Instalación (clona el repositorio de GitHub)

In [ ]:
REPO = 'https://github.com/diarce/Auditoria.git'   # ajustar si el repositorio es otro
!rm -rf /content/Auditoria && git clone -q $REPO /content/Auditoria
%cd /content/Auditoria
!pip install -q -r requirements.txt
!playwright install --with-deps chromium > /dev/null 2>&1 && echo 'Chromium listo'

## 2. Parámetros
El **contacto** viaja en el agente de usuario (exigencia del protocolo ético). La **semilla** define los alias: guardala en un lugar seguro para poder reproducir la anonimización.

In [ ]:
from auditoria import Config
cfg = Config(
    CONTACTO='TU_CORREO@unam.edu.ar',        # <- obligatorio: identifica al agente ante los sitios
    SALT_ANONIMIZACION='UNA-SEMILLA-PROPIA', # <- cambiar
    DIR_SALIDA='/content/resultados',
    ACEPTAR_MAYORISTA_DECLARADA=True,        # E6: decisión metodológica abierta (ver README)
)
assert 'CONFIGURAR' not in cfg.CONTACTO and 'TU_CORREO' not in cfg.CONTACTO, 'Completá el contacto'
print(cfg.user_agent())

## 3. Ejecución
Usa Google Maps **solo si** el `robots.txt` lo permite; si no, se detiene sin eludir nada y se puede usar el respaldo por CSV (celda siguiente).
Con ~90 sitios la auditoría demora aproximadamente 1–2 horas por las demoras éticas entre peticiones.

In [ ]:
from auditoria.pipeline import ejecutar
from auditoria.fuentes import FuenteBloqueada
try:
    resultado = ejecutar(cfg, fuente='maps')
except FuenteBloqueada as e:
    print('FUENTE BLOQUEADA POR PROTOCOLO:', e)
    resultado = None

### 3b. Respaldo: lista propia (CSV con columnas `web`, opcional `direccion`, `categoria`)

In [ ]:
if resultado is None:
    from google.colab import files
    subido = files.upload()                      # elegí tu CSV
    resultado = ejecutar(cfg, fuente='csv', ruta_csv=list(subido)[0])

## 4. Informe

In [ ]:
from IPython.display import HTML, display
display(HTML(open(f'{cfg.DIR_SALIDA}/informe_auditoria.html', encoding='utf-8').read()))

## 5. Verificación manual (7 indicadores)
Completá `manual_plantilla.csv` (columna `puntaje_0a3`) siguiendo el protocolo manual, subilo y regenerá el informe.

In [ ]:
from auditoria.pipeline import aplicar_manual
from google.colab import files
subido = files.upload()
aplicar_manual(cfg, resultado, list(subido)[0])
print('Informe regenerado')

## 6. Descargar resultados
`mapeo_privado.csv` vincula alias y dominio: **confidencial**, no subir a GitHub ni publicar.

In [ ]:
!cd /content && zip -qr resultados.zip resultados
from google.colab import files
files.download('/content/resultados.zip')